# Categorical AR-HMM: simulate and fit

Two-level generative model:

| Level | Variable | Description |
|-------|----------|-------------|
| Outer | $z_t$ | Behavioral mode (e.g. escape, exploration) — slow, **latent** |
| Obs   | $c_t$ | Syllable ID — fast, **observed**; transitions are mode-specific |

**Generative story** for each bout $t$:
1. Transition behavioral mode: $z_t \mid z_{t-1} \sim A^{\text{out}}[z_{t-1}, :]$
2. Transition syllable: $c_t \mid c_{t-1},\, z_t \sim A^{\text{in}}_{z_t}[c_{t-1}, :]$

$A^{\text{in}}_{z_t}$ is a $K_{\text{obs}} \times K_{\text{obs}}$ transition matrix over syllable types, specific to mode $z_t$.  
The mode **switches the Markov dynamics** of the observed sequence — this is a **categorical AR-HMM** (discrete analogue of a switching AR model / sLDS without a separate latent state).

**Fitting**: EM with a custom `ssm` observation class.
- E-step: forward–backward on $K_{\text{out}}$ states with log-likelihood $\ell_t(z) = \log A^{\text{in}}_z[c_{t-1},\, c_t]$
- M-step: $A^{\text{out}}$ from standard HMM sufficient statistics; $A^{\text{in}}_z$ from weighted syllable transition counts under mode $z$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ssm
from ssm.observations import Observations
from itertools import permutations as _perms

plt.rcParams.update({'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False})
np.random.seed(0)

K_out, K_obs = 2, 5   # behavioral modes, syllable types
n_bouts = 500

# ── custom observation class ──────────────────────────────────────────────────
class CategoricalARObservations(Observations):
    """
    Categorical autoregressive observations for a switching HMM.

    At each time step t > 0:
        P(c_t | c_{t-1}, z_t) = A_in[z_t, c_{t-1}, c_t]

    A_in[k] is a (C x C) syllable transition matrix specific to mode k,
    stored in log-space as self.logAs[k].
    For t = 0 the likelihood is uniform (no previous observation).
    """

    def __init__(self, K, D, M=0, C=2):
        super().__init__(K, D, M)
        self.C = C
        self.logAs = np.full((K, C, C), -np.log(C))   # log A_in, shape (K, C, C)

    @property
    def params(self):
        return (self.logAs,)

    @params.setter
    def params(self, value):
        self.logAs, = value

    def initialize(self, datas, inputs=None, masks=None, tags=None, **kwargs):
        self.logAs = np.log(
            np.array([[np.random.dirichlet(np.ones(self.C))
                       for _ in range(self.C)] for _ in range(self.K)]) + 1e-8
        )

    def log_likelihoods(self, data, input, mask, tag):
        """
        Return (T, K) array of per-state log-likelihoods.

        For t = 0 : uniform log-likelihood -log(C) for all states.
        For t > 0 : log A_in[k, c[t-1], c[t]]  for each mode k.

        Hint: self.logAs has shape (K, C, C).
              self.logAs[:, c[:-1], c[1:]] selects the right entry for every
              timestep at once and gives shape (K, T-1) — transpose to (T-1, K).
        """
        c = data[:, 0].astype(int)
        lls = np.full((len(c), self.K), -np.log(self.C))   # t=0: uniform
        # ── YOUR CODE HERE ────────────────────────────────────────────────────
        # lls[1:] = ...
        return lls

    def sample(self, z, input=None, tag=None, with_noise=True):
        C, T = self.C, len(z)
        c = np.zeros(T, int)
        c[0] = np.random.choice(C)
        for t in range(1, T):
            p = np.exp(self.logAs[z[t], c[t-1]])
            c[t] = np.random.choice(C, p=p / p.sum())
        return c[:, None]

    def m_step(self, expectations, datas, inputs, masks, tags, **kwargs):
        """
        Update self.logAs from the E-step posteriors.

        For each time step t = 1, ..., T-1, accumulate the soft count of
        mode k being active during the syllable transition c[t-1] -> c[t]:

            counts[k, c[t-1], c[t]] += E[z_t = k]

        i.e. for all k simultaneously:

            counts[:, c[t-1], c[t]] += Ez[t]     # Ez[t] has shape (K,)

        Then normalise each row to get the updated transition matrix:

            A_in[k, p, :] = counts[k, p, :] / sum_n counts[k, p, n]
            logAs[k, p, :] = log A_in[k, p, :]
        """
        counts = np.zeros((self.K, self.C, self.C))
        for (Ez, _, _), data in zip(expectations, datas):
            c = data[:, 0].astype(int)
            # ── YOUR CODE HERE ────────────────────────────────────────────────
            # for t in range(len(c) - 1):
            #     counts[:, c[t], c[t+1]] += ...
        counts += 1e-8
        self.logAs = np.log(counts) - np.log(counts.sum(-1, keepdims=True))

# ── true parameters ───────────────────────────────────────────────────────────
log_Ps_true = np.log(np.array([[0.90, 0.10],
                                [0.10, 0.90]]))
W_true = np.array([[2.0], [-2.0]])   # high u → mode 0, low u → mode 1

# mode 0: forward cycle 0→1→2→3→4→0 ;  mode 1: backward 0→4→3→2→1→0
eps = 0.20 / (K_obs - 1)
A_in_true = np.full((K_out, K_obs, K_obs), eps)
for s in range(K_obs):
    A_in_true[0, s, (s + 1) % K_obs] = 0.80
    A_in_true[1, s, (s - 1) % K_obs] = 0.80

# ── continuous input: slow random walk ───────────────────────────────────────
u_obs = np.cumsum(0.1 * np.random.randn(n_bouts))
u_obs = (u_obs - u_obs.mean()) / u_obs.std()

# ── simulate ──────────────────────────────────────────────────────────────────
z_true = np.zeros(n_bouts, int)
c_obs  = np.zeros(n_bouts, int)

p0 = np.exp(log_Ps_true[0] + u_obs[0] * W_true[:, 0])
z_true[0] = np.random.choice(K_out, p=p0 / p0.sum())
c_obs[0]  = np.random.choice(K_obs)

for t in range(1, n_bouts):
    # Sample z_true[t] from the input-driven transition:
    #   log p(z' | z, u) ∝ log_Ps_true[z, z'] + u * W_true[z']
    # ── YOUR CODE HERE ────────────────────────────────────────────────────────
    # log_p = ...
    # z_true[t] = ...
    # c_obs[t]  = ...

print(f'{n_bouts} bouts | outer state counts {np.bincount(z_true)}')
print(f'Syllable counts: {np.bincount(c_obs)}')

In [ ]:
# ── Fit the model ─────────────────────────────────────────────────────────────
# 1. Create an ssm.HMM with K_out states, 1-D observation, 1-D input,
#    and inputdriven transitions.
# 2. Attach a CategoricalARObservations instance as model.observations.
# 3. Fit with EM (num_iters=100); store the log-likelihood trace in `lls`.
# 4. Print the initial and final log-likelihood.

# YOUR CODE HERE

In [ ]:
# ── Decode and evaluate ────────────────────────────────────────────────────────
# 1. Recover the most likely state sequence with model.most_likely_states().
# 2. Find the best permutation of states matching z_hat to z_true (label switching).
# 3. Print outer state accuracy.
# 4. Extract A_out_fit, A_in_fit, W_fit (re-ordered by best_perm).
# 5. Compare true vs fitted W, and plot A_out and A_in matrices side by side.

# YOUR CODE HERE

In [ ]:
# ── Plot timeseries ────────────────────────────────────────────────────────────
# Show (from top to bottom, shared x-axis):
#   1. input u_obs
#   2. true latent state z_true
#   3. inferred state z_hat (best permutation)

# YOUR CODE HERE